для запуска нужны python 3.11

```bash
pip install numpy==2.4.6 pandas==3.0.5 scipy==1.17.1 scikit-learn==1.9.1 nltk==3.10.3 pyarrow==25.0.1 tqdm==4.70.1 catboost==1.2.10 torch==2.7.1 transformers==4.51.3 ipykernel==7.3.0
```

```python
from huggingface_hub import snapshot_download
snapshot_download(
    'intfloat/multilingual-e5-small',
    revision='614241f622f53c4eeff9890bdc4f31cfecc418b3',
    local_dir='models/multilingual-e5-small',
    allow_patterns=['*.json', 'model.safetensors', 'sentencepiece.bpe.model'],
)
```

готовые методы и модели: bm25, tf idf из [scikit learn](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction), русский snowball из [nltk](https://www.nltk.org/api/nltk.stem.snowball.html), [catboost](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier), [multilingual e5 small](https://huggingface.co/intfloat/multilingual-e5-small)


импортируем все нужное и прочитаем данные. кроме текста и локации возьмем подкатегорию, цену, рейтинг и отзывы. дальше проверим, помогут ли они выбрать объявления, когда по тексту подходит сразу много вариантов


In [1]:
import gc
import re
import time
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.stem.snowball import RussianStemmer
from tqdm import tqdm
from catboost import CatBoostClassifier

started = time.time()
data_dir = Path('data')
search_cols = [
    'search_query', 'search_location_id', 'search_is_delivery_search',
    'search_infm_params_text', 'search_category',
]
item_cols = [
    'item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text',
    'item_location_id', 'item_latitude', 'item_longitude',
    'item_microcat_id', 'item_price', 'item_rating', 'item_rating_reviews_count',
    'item_is_phone_hidden', 'item_is_message_forbidden',
]

train = pd.read_parquet(data_dir / 'train.parquet', columns=search_cols + item_cols)
queries = pd.read_parquet(data_dir / 'benchmark_queries.parquet')
benchmark_items = pd.read_parquet(data_dir / 'benchmark_items.parquet', columns=item_cols)
print('строк train:', len(train))
print('запросов benchmark:', len(queries))
print('объявлений benchmark:', len(benchmark_items))
print('совпадение локаций в train:', f'{(train.search_location_id == train.item_location_id).mean():.1%}')

строк train: 497673
запросов benchmark: 2452
объявлений benchmark: 189212
совпадение локаций в train: 83.1%


сначала подготовим текст: переведем его в нижний регистр, уберем знаки препинания и лишние пробелы. для поиска по словам будем обрезать окончания с помощью стемминга. так разные формы одного слова чаще будут совпадать при поиске

уберем повторяющиеся пары запроса и объявления. затем соберем объявления из train и benchmark в одну таблицу, чтобы искать среди них при проверке решения. каждый item_id оставим один раз. если объявление есть в обоих файлах, возьмем его данные из benchmark

еще нам понадобится примерный центр каждой локации. возьмем медиану широты и долготы ее объявлений. дальше от этой точки будем считать расстояние до объявления


In [2]:
def clean(text):
    text = str(text).lower().replace('\u0451', 'е').replace('\\n', ' ')
    return ' '.join(re.findall(r'[а-яa-z0-9]+', text))

stemmer = RussianStemmer()

@lru_cache(maxsize=150000)
def stem(word):
    return stemmer.stem(word)


def tokenize(text):
    return [stem(word) for word in clean(text).split() if len(word) > 1]


train['query_text'] = train.search_query.map(clean)
interactions = train[search_cols + ['query_text', 'item_id']].drop_duplicates().copy()
all_items = pd.concat([train[item_cols], benchmark_items], ignore_index=True)
all_items = all_items.drop_duplicates('item_id', keep='last').reset_index(drop=True)
del train
gc.collect()


all_items[['item_latitude', 'item_longitude']] = all_items[['item_latitude', 'item_longitude']].astype(float)
centers = all_items.groupby('item_location_id')[['item_latitude', 'item_longitude']].median()

known_queries = set(interactions.query_text)
print('новых текстов в benchmark:', f'{(~queries.search_query.map(clean).isin(known_queries)).mean():.1%}')
print('объявлений в корпусе для проверки:', len(all_items))
all_items['item_price'] = all_items.item_price.astype(float)
benchmark_items['item_price'] = benchmark_items.item_price.astype(float)
item_metadata = all_items[['item_id', 'item_location_id', 'item_microcat_id']].copy()

новых текстов в benchmark: 62.5%
объявлений в корпусе для проверки: 515895


сначала сохраним прежние 5000 запросов для обучения отбора и добавим еще 10000 новых текстов. для сравнения настроек оставим прежние 1800 запросов. отдельно выберем 2000 новых текстов для последней проверки

из истории поиска уберем все запросы, которые участвуют в обучении отбора или проверке. пересчитаем признаки по оставшейся истории. это важно: дополнительные 10000 запросов раньше были частью истории, поэтому без пересчета их ответы могли бы попасть в признаки

первые 5000 запросов оставим в прежнем порядке. обе модели будут учиться на одних и тех же примерах для этой части, а модель на 15000 получит еще 10000


In [3]:
rng = np.random.default_rng(42)
texts = interactions.query_text.unique()
first_texts = rng.choice(texts, 2400, replace=False)
remaining = texts[~pd.Index(texts).isin(first_texts)]
previous_test_texts = rng.choice(remaining, 1200, replace=False)
used_texts = set(first_texts) | set(previous_test_texts)

used_rows = interactions[interactions.query_text.isin(used_texts)]
used_queries = used_rows.groupby(search_cols, sort=False, dropna=False).agg(
    relevant=('item_id', lambda values: set(values)),
    query_text=('query_text', 'first'),
).reset_index()
used_queries = used_queries.sample(frac=1, random_state=42).drop_duplicates('query_text')

first_part = used_queries[used_queries.query_text.isin(first_texts)].sample(n=600, random_state=42)
second_part = used_queries[used_queries.query_text.isin(previous_test_texts)].sample(frac=1, random_state=42)
validation = pd.concat([second_part, first_part], ignore_index=True)

remaining = texts[~pd.Index(texts).isin(used_texts)]
test_texts = np.random.default_rng(314).choice(remaining, 1200, replace=False)
test_rows = interactions[interactions.query_text.isin(test_texts)]
test = test_rows.groupby(search_cols, sort=False, dropna=False).agg(
    relevant=('item_id', lambda values: set(values)),
    query_text=('query_text', 'first'),
).reset_index()
test = test.sample(frac=1, random_state=42).drop_duplicates('query_text')
test = test.sample(frac=1, random_state=43).reset_index(drop=True)

held_texts = used_texts | set(test_texts)
fit = interactions[~interactions.query_text.isin(held_texts)].copy()
assert not set(fit.query_text) & held_texts
assert not set(validation.query_text) & set(test.query_text)
print('обучающих пар:', len(fit))
print('для сравнения вариантов:', len(validation))
print('для итоговой проверки:', len(test))
rank_texts = np.random.default_rng(2026).choice(fit.query_text.unique(), 5000, replace=False)
rank_rows = fit[fit.query_text.isin(rank_texts)]
rank_queries = rank_rows.groupby(search_cols, sort=False, dropna=False).agg(
    relevant=('item_id', lambda values: set(values)), query_text=('query_text', 'first')
).reset_index().sample(frac=1, random_state=42).drop_duplicates('query_text').reset_index(drop=True)
fit = fit[~fit.query_text.isin(rank_texts)].copy()
assert not set(rank_queries.query_text) & set(fit.query_text)
assert not set(rank_queries.query_text) & held_texts
fit = fit.merge(item_metadata, on='item_id', how='left', validate='many_to_one')
interactions = interactions.merge(item_metadata, on='item_id', how='left', validate='many_to_one')
print('пар в истории поиска:', len(fit))
print('запросов для обучения отбора:', len(rank_queries))
old_rank_queries = rank_queries.copy()
remaining_texts = fit.query_text.unique()
fresh_texts = np.random.default_rng(20260917).choice(remaining_texts, 2000, replace=False)
remaining_texts = remaining_texts[~pd.Index(remaining_texts).isin(fresh_texts)]
extra_texts = np.random.default_rng(20260918).choice(remaining_texts, 10000, replace=False)


def query_contexts(rows):
    return rows.groupby(search_cols, sort=False, dropna=False).agg(
        relevant=('item_id', lambda values: set(values)), query_text=('query_text', 'first')
    ).reset_index().sample(frac=1, random_state=42).drop_duplicates('query_text').reset_index(drop=True)


fresh = query_contexts(fit[fit.query_text.isin(fresh_texts)])
extra_queries = query_contexts(fit[fit.query_text.isin(extra_texts)])
rank_queries = pd.concat([old_rank_queries, extra_queries], ignore_index=True)
fit = fit[~fit.query_text.isin(set(fresh_texts) | set(extra_texts))].copy()
for frame in [rank_queries, validation, test, fresh]:
    assert not set(frame.query_text) & set(fit.query_text)
assert rank_queries.iloc[:5000].equals(old_rank_queries)
assert not set(fresh.query_text) & set(rank_queries.query_text)
print('пар в истории:', len(fit))
print('запросов для обучения:', len(rank_queries))
print('запросов для новой проверки:', len(fresh))

обучающих пар: 441978
для сравнения вариантов: 1800
для итоговой проверки: 1200


пар в истории поиска: 414484
запросов для обучения отбора: 5000


пар в истории: 338756
запросов для обучения: 15000
запросов для новой проверки: 2000


добавим поиск по смыслу через multilingual e5 small. она превращает текст в набор чисел. похожие по смыслу тексты должны получать похожие наборы, даже если сами слова различаются

для объявления возьмем заголовок и начало параметров до слов «место оказания услуг». адреса и расписание в этот текст не добавляем. для запроса оставим его исходный текст. перед объявлением добавим passage:, а перед запросом query:, как указано в описании модели

веса читаем из локальной папки models. как их один раз скачать, указано в конце ноутбука. во время расчета обращений к внешним api нет

векторы сохраним в папку cache, чтобы не считать их при каждом запуске. одинаковые тексты объявлений обработаем один раз, а потом вернем результат к исходному порядку item_id


In [4]:
cache_dir = Path('cache')
cache_dir.mkdir(exist_ok=True)
item_path = cache_dir / 'item_embeddings.npy'
query_path = cache_dir / 'query_embeddings.npy'
query_texts = pd.concat([
    frame.search_query for frame in [rank_queries, validation, test, fresh, queries]
], ignore_index=True).drop_duplicates().reset_index(drop=True).to_frame('search_query')
need_encoder = not item_path.exists() or not query_path.exists()
if need_encoder:
    import torch
    from transformers import AutoModel, AutoTokenizer
    torch.set_num_threads(2)
    model_dir = Path('models/multilingual-e5-small')
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    encoder = AutoModel.from_pretrained(model_dir, local_files_only=True).to(device).eval()


def encode_texts(texts, prefix, path):
    texts = [prefix + text for text in texts]
    lengths = np.array([len(ids) for ids in tokenizer(texts, truncation=True, max_length=96)['input_ids']])
    order = np.argsort(lengths)
    output = np.lib.format.open_memmap(path, mode='w+', dtype=np.float32, shape=(len(texts), 384))
    with torch.inference_mode():
        for left in tqdm(range(0, len(texts), 64), desc='смысл', mininterval=20):
            positions = order[left:left + 64]
            batch = tokenizer(
                [texts[i] for i in positions], truncation=True, max_length=96,
                padding=True, return_tensors='pt',
            ).to(device)
            hidden = encoder(**batch).last_hidden_state
            pooled = hidden.masked_fill(~batch['attention_mask'][..., None].bool(), 0).sum(1)
            pooled /= batch['attention_mask'].sum(1)[:, None]
            output[positions] = torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy()
    output.flush()
    return output


if not item_path.exists():
    texts = all_items.item_title_raw.fillna('').str.strip() + '. '
    texts += all_items.item_infm_params_text.fillna('').str.split('Место оказания услуг').str[0].str[:300].str.strip()
    codes, unique_texts = pd.factorize(texts, sort=False)
    unique_vectors = encode_texts(unique_texts, 'passage: ', cache_dir / 'unique_embeddings.npy')
    output = np.lib.format.open_memmap(item_path, mode='w+', dtype=np.float32, shape=(len(all_items), 384))
    for left in range(0, len(codes), 8192):
        output[left:left + 8192] = unique_vectors[codes[left:left + 8192]]
    output.flush()
    all_items[['item_id']].to_parquet(cache_dir / 'item_ids.parquet', index=False)
    del output, unique_vectors, texts, unique_texts, codes
if not query_path.exists():
    output = encode_texts(query_texts.search_query, 'query: ', query_path)
    query_texts.to_parquet(cache_dir / 'query_texts.parquet', index=False)
    del output
if need_encoder:
    del encoder, tokenizer
    if device == 'mps':
        torch.mps.empty_cache()

assert np.array_equal(pd.read_parquet(cache_dir / 'item_ids.parquet').item_id, all_items.item_id)
assert np.array_equal(pd.read_parquet(cache_dir / 'query_texts.parquet').search_query, query_texts.search_query)
item_vectors = np.load(item_path, mmap_mode='r')
query_vectors = np.load(query_path, mmap_mode='r')
query_index = pd.Index(query_texts.search_query)
item_index = pd.Index(all_items.item_id)
print('векторы объявлений:', item_vectors.shape)
print('векторы запросов:', query_vectors.shape)

векторы объявлений: (515895, 384)
векторы запросов: (22215, 384)


соберем текст объявления из заголовка, параметров, описания и запросов из обучающей истории. заголовок повторим три раза, запросы из истории два раза. так слова из этих частей будут сильнее влиять на оценку

возьмем до восьми разных запросов на объявление, первые 400 символов параметров и первые 4500 символов описания. слова, которые встретились только один раз, тоже оставим. эти настройки выбрали в предыдущих экспериментах

по полученным текстам построим bm25. он оценивает совпадения слов, дает больший вес редким словам и учитывает длину текста


In [5]:
def build_word_index(items, history, description_length=4500, min_df=1, max_features=350000):
    known_queries = history.groupby('item_id').query_text.agg(
        lambda values: ' '.join(pd.unique(values)[:8])
    )
    documents = (items.item_title_raw.fillna('') + ' ') * 3
    documents += items.item_infm_params_text.fillna('').str[:400] + ' '
    documents += items.item_description_raw.fillna('').str[:description_length] + ' '
    documents += (items.item_id.map(known_queries).fillna('') + ' ') * 2

    vectorizer = CountVectorizer(
        tokenizer=tokenize, token_pattern=None, lowercase=False,
        min_df=min_df, max_df=0.9, max_features=max_features, dtype=np.float32,
    )
    counts = vectorizer.fit_transform(tqdm(documents, desc='слова', mininterval=20))
    lengths = np.asarray(counts.sum(axis=1)).ravel()
    document_frequency = np.bincount(counts.indices, minlength=counts.shape[1])
    idf = np.log1p((len(items) - document_frequency + 0.5) / (document_frequency + 0.5))

    k1 = 1.5
    b = 0.75
    length_norm = k1 * (1 - b + b * lengths / lengths.mean())
    entries = counts.tocoo()
    values = entries.data.copy()
    values *= (k1 + 1) / (entries.data + length_norm[entries.row])
    values *= idf[entries.col]
    word_index = sparse.csr_matrix(
        (values, (entries.col, entries.row)), shape=(counts.shape[1], len(items))
    )
    return vectorizer, word_index

к поиску по всему тексту добавим отдельные оценки по заголовку. одна будет учитывать совпадения слов, другая совпадения кусочков слов длиной от 3 до 5 символов. кусочки могут совпасть даже при опечатке или другом окончании

еще найдем похожие запросы в истории. для каждого из них известно, объявления каких подкатегорий выбирали пользователи. возьмем 30 самых похожих запросов и сложим их доли по подкатегориям. сходство возведем в четвертую степень, чтобы близкие формулировки влияли сильнее случайных совпадений

по этой же истории посчитаем связь локации поиска с локацией выбранного объявления. все это соберем в одной функции, потому что перед итоговым ответом поиск нужно будет пересобрать для benchmark_items


In [6]:
def build_search(items, history):
    words, word_index = build_word_index(items, history)
    chars = TfidfVectorizer(
        preprocessor=clean, analyzer='char_wb', ngram_range=(3, 5),
        min_df=3, max_features=180000, sublinear_tf=True, dtype=np.float32,
    )
    char_index = chars.fit_transform(items.item_title_raw.fillna('')).T.tocsr()

    title_counts = words.transform(items.item_title_raw.fillna(''))
    title_binary = title_counts.copy()
    title_binary.data[:] = 1
    lengths = np.asarray(title_counts.sum(axis=1)).ravel()
    df = np.bincount(title_counts.indices, minlength=title_counts.shape[1])
    idf = np.log1p((len(items) - df + 0.5) / (df + 0.5))
    length_norm = 1.5 * (0.25 + 0.75 * lengths / max(lengths.mean(), 1e-6))
    title_counts.data *= 2.5 / (title_counts.data + np.repeat(length_norm, np.diff(title_counts.indptr)))
    title_counts.data *= idf[title_counts.indices]
    title_index = title_counts.T.tocsr()

    query_categories = history.groupby(['query_text', 'item_microcat_id']).size().unstack(fill_value=0)
    query_categories = query_categories.div(query_categories.sum(axis=1), axis=0)
    query_vectorizer = TfidfVectorizer(
        preprocessor=clean, analyzer='char_wb', ngram_range=(3, 5), min_df=2,
        max_features=100000, sublinear_tf=True, dtype=np.float32,
    )
    query_text_index = query_vectorizer.fit_transform(query_categories.index).T.tocsr()
    query_category_matrix = sparse.csr_matrix(query_categories.to_numpy(dtype=np.float32))
    counts = history.groupby(['search_location_id', 'item_location_id']).size()
    location_probabilities = counts / counts.groupby(level=0).transform('sum')
    return {
        'items': items, 'fit': history, 'centers': centers,
        'word_vectorizer': words, 'word_index': word_index,
        'char_vectorizer': chars, 'char_index': char_index,
        'title_index': title_index, 'title_binary': title_binary.T.tocsr(),
        'location_probabilities': location_probabilities,
        'query_vectorizer': query_vectorizer, 'query_text_index': query_text_index,
        'query_category_matrix': query_category_matrix, 'category_ids': query_categories.columns.to_numpy(),
    }

search = build_search(all_items, fit)


слова:   0%|          | 0/515895 [00:00<?, ?it/s]


слова:  14%|█▍        | 72750/515895 [00:20<02:01, 3637.45it/s]


слова:  14%|█▍        | 72750/515895 [00:30<02:01, 3637.45it/s]


слова:  30%|██▉       | 153047/515895 [00:40<01:34, 3859.44it/s]


слова:  30%|██▉       | 153047/515895 [00:50<01:34, 3859.44it/s]

...


слова:  74%|███████▍  | 384286/515895 [01:40<00:34, 3826.05it/s]


слова:  74%|███████▍  | 384286/515895 [01:50<00:34, 3826.05it/s]


слова:  88%|████████▊ | 456463/515895 [02:00<00:15, 3752.19it/s]


слова:  88%|████████▊ | 456463/515895 [02:10<00:15, 3752.19it/s]


слова: 100%|██████████| 515895/515895 [02:17<00:00, 3765.35it/s]

сначала соберем прежний набор кандидатов: 500 по общей оценке, 200 по заголовку, 200 с учетом подкатегории и 100 только по тексту. повторы уберем

затем добавим 100 объявлений с самым большим сходством по смыслу и еще 300 с учетом смысла и локации. сходство считаем как скалярное произведение нормированных векторов. для второго списка прибавим к нему 0.05 от логарифма прежнего множителя за локацию. так близость будет помогать, но не перекроет полностью смысл запроса

к прежним 30 признакам добавим сходство по смыслу и его оценку с учетом локации. место старого кандидата в прежнем списке сохраним. для нового возьмем число старых кандидатов, у которых выше исходная оценка

для обучения отдельно выберем примеры из старого и расширенного списков. в каждом оставим первые 200, еще 200 случайных и все найденные правильные ответы. признаки посчитаем для объединения этих примеров, а масками отметим, какие строки нужны каждому варианту. это позволит сравнить оба подхода без повторного расчета всех признаков

при проверке используем полные списки кандидатов. правильные ответы нужны только для расчета метрики, в сам поиск их не добавляем


In [7]:
def top_indices(scores, count):
    return np.argpartition(scores, -count)[-count:]


def make_features(cache, frame, name, item_vectors, query_vectors, training=False):
    items = cache['items']
    ids = items.item_id.to_numpy()
    query_words = cache['word_vectorizer'].transform(frame.search_query).tocsr()
    query_words.data[:] = 1
    query_filters = cache['word_vectorizer'].transform(frame.search_infm_params_text).tocsr()
    query_filters.data[:] = 1
    query_chars = cache['char_vectorizer'].transform(frame.search_query).tocsr()
    neighbor_chars = cache['query_vectorizer'].transform(frame.search_query).tocsr()
    locations = items.item_location_id.to_numpy()
    city_ids, city_codes = np.unique(locations, return_inverse=True)
    probabilities = cache['location_probabilities']
    known_locations = set(probabilities.index.get_level_values(0))
    centers = cache['centers']
    latitude = np.radians(items.item_latitude.to_numpy(dtype=np.float32))
    longitude = np.radians(items.item_longitude.to_numpy(dtype=np.float32))
    clean_titles = items.item_title_raw.fillna('').map(clean).to_numpy()
    microcat_codes = pd.Index(cache['category_ids']).get_indexer(items.item_microcat_id)
    popularity = cache['fit'].item_id.value_counts()
    filter_rows = cache['fit']
    filter_counts = filter_rows.groupby(['search_infm_params_text', 'item_microcat_id']).size()
    filter_probabilities = filter_counts / filter_counts.groupby(level=0).transform('sum')
    known_filters = set(filter_probabilities.index.get_level_values(0))
    microcats = items.item_microcat_id.to_numpy()
    del filter_rows
    fixed = np.column_stack([
        np.log1p(items.item_price.fillna(0).clip(lower=0)),
        items.item_rating.fillna(0),
        np.log1p(items.item_rating_reviews_count.fillna(0).clip(lower=0)),
        items.item_is_phone_hidden.fillna(False),
        items.item_is_message_forbidden.fillna(False),
        np.log1p(items.item_title_raw.fillna('').str.len()),
        np.log1p(items.item_description_raw.fillna('').str.len()),
        np.log1p(items.item_id.map(popularity).fillna(0)),
        items.item_price.lt(0),
        items.item_rating.isna(),
    ]).astype(np.float32)
    names = [
        'word', 'char', 'filters', 'title', 'title_coverage', 'word_raw', 'title_raw',
        'text_score', 'baseline_score', 'same_location', 'nearby', 'location_probability',
        'category_probability', 'category_similarity', 'neighbor_similarity', 'query_words',
        'query_chars', 'title_phrase', 'baseline_rank', 'filter_category', 'price', 'rating', 'reviews',
        'phone_hidden', 'message_forbidden', 'title_length', 'description_length', 'popularity', 'negative_price', 'missing_rating',
    ]
    names += ['dense', 'dense_geo']
    blocks, labels, candidates, offsets = [], [], [], [0]
    lexical_masks, hybrid_masks = [], []
    hybrid_rng = np.random.default_rng(142)
    rng = np.random.default_rng(42)
    baseline_recalls, pool_recalls = [], []
    for i, row in enumerate(tqdm(frame.itertuples(index=False), total=len(frame), desc=name, mininterval=20)):
        if i % 32 == 0:
            dense_batch = query_vectors[i:i + 32] @ item_vectors.T
        dense = dense_batch[i % 32]
        word_raw = (query_words[i] @ cache['word_index']).toarray().ravel()
        char = (query_chars[i] @ cache['char_index']).toarray().ravel()
        filters = (query_filters[i] @ cache['word_index']).toarray().ravel()
        title_raw = (query_words[i] @ cache['title_index']).toarray().ravel()
        word = word_raw / max(word_raw.max(), 1e-9)
        char /= max(char.max(), 1e-9)
        filters /= max(filters.max(), 1e-9)
        title = title_raw / max(title_raw.max(), 1e-9)
        coverage = (query_words[i] @ cache['title_binary']).toarray().ravel()
        coverage /= max(query_words[i].nnz, 1)

        same = locations == row.search_location_id
        nearby = np.zeros(len(items), dtype=np.float32)
        if row.search_location_id in centers.index:
            qlat, qlon = np.radians(centers.loc[row.search_location_id].to_numpy())
            a = np.sin((latitude - qlat) / 2) ** 2
            a += np.cos(qlat) * np.cos(latitude) * np.sin((longitude - qlon) / 2) ** 2
            distance = 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
            nearby = np.nan_to_num(np.exp(-distance / 40), nan=0).astype(np.float32) * (~same)
        city_probability = np.zeros(len(items), dtype=np.float32)
        if row.search_location_id in known_locations:
            by_city = probabilities.loc[row.search_location_id].reindex(city_ids, fill_value=0).to_numpy(dtype=np.float32)
            city_probability = by_city[city_codes]

        similarities = (neighbor_chars[i] @ cache['query_text_index']).toarray().ravel()
        neighbors = top_indices(similarities, min(30, len(similarities)))
        weights = similarities[neighbors] ** 4
        category_distribution = np.asarray(weights @ cache['query_category_matrix'][neighbors]).ravel()
        category_distribution /= max(category_distribution.sum(), 1e-9)
        best_similarities = cache['query_category_matrix'][neighbors].copy()
        best_similarities.data[:] = 1
        best_similarities = best_similarities.multiply(similarities[neighbors, None]).max(axis=0).toarray().ravel()
        category_probability = np.where(microcat_codes >= 0, category_distribution[microcat_codes], 0)
        category_similarity = np.where(microcat_codes >= 0, best_similarities[microcat_codes], 0)
        text_score = 0.7 * word + 0.3 * char + 0.05 * filters
        geo = 1 + 7 * same + 3 * nearby + 3 * np.sqrt(city_probability)
        score = text_score * geo
        lexical_pool = np.unique(np.concatenate([
            top_indices(score, 500),
            top_indices((0.7 * title + 0.3 * char) * geo, 200),
            top_indices(score * (0.05 + category_probability), 200),
            top_indices(text_score, 100),
        ]))
        lexical_pool = lexical_pool[np.argsort(-score[lexical_pool], kind='stable')]
        dense_geo = dense + 0.05 * np.log(geo)
        pool = np.unique(np.concatenate([
            lexical_pool, top_indices(dense, 100), top_indices(dense_geo, 300),
        ]))
        pool = pool[np.argsort(-score[pool], kind='stable')]
        is_lexical = np.isin(pool, lexical_pool)
        is_hybrid = np.ones(len(pool), dtype=bool)
        rank = np.searchsorted(-score[lexical_pool], -score[pool]).astype(np.int32)
        lexical_order = np.argsort(lexical_pool)
        rank[is_lexical] = lexical_order[np.searchsorted(lexical_pool[lexical_order], pool[is_lexical])]
        relevant = row.relevant if hasattr(row, 'relevant') else set()
        y = np.isin(ids[pool], list(relevant))
        if relevant:
            baseline_recalls.append(len(set(ids[top_indices(score, 50)]) & relevant) / len(relevant))
            pool_recalls.append(y.sum() / len(relevant))
        if training:
            lexical_positions = np.flatnonzero(is_lexical)
            lexical_keep = lexical_positions[np.unique(np.concatenate([
                np.arange(200), np.flatnonzero(y[lexical_positions]),
                rng.choice(np.arange(200, len(lexical_positions)), 200, replace=False),
            ]))]
            hybrid_keep = np.unique(np.concatenate([
                np.arange(200), np.flatnonzero(y),
                hybrid_rng.choice(np.arange(200, len(pool)), 200, replace=False),
            ]))
            keep = np.union1d(lexical_keep, hybrid_keep)
            is_lexical = np.isin(keep, lexical_keep)
            is_hybrid = np.isin(keep, hybrid_keep)
            pool, y, rank = pool[keep], y[keep], rank[keep]
        lexical_masks.append(is_lexical)
        hybrid_masks.append(is_hybrid)
        qtext = clean(row.search_query)
        filter_category = np.zeros(len(pool), dtype=np.float32)
        if row.search_infm_params_text and row.search_infm_params_text in known_filters:
            filter_category = filter_probabilities.loc[row.search_infm_params_text].reindex(
                microcats[pool], fill_value=0
            ).to_numpy(dtype=np.float32)
        dynamic = np.column_stack([
            word[pool], char[pool], filters[pool], title[pool], coverage[pool],
            word_raw[pool], title_raw[pool], text_score[pool], score[pool], same[pool],
            nearby[pool], city_probability[pool], category_probability[pool], category_similarity[pool],
            np.full(len(pool), similarities.max()), np.full(len(pool), query_words[i].nnz),
            np.full(len(pool), len(qtext)), [float(qtext in t) for t in clean_titles[pool]], np.log1p(rank), filter_category,
        ]).astype(np.float32)
        blocks.append(np.column_stack([dynamic, fixed[pool], dense[pool], dense_geo[pool]]))
        labels.append(y)
        candidates.append(pool.astype(np.int32))
        offsets.append(offsets[-1] + len(pool))
    denominators = np.array([
        len(row.relevant) if hasattr(row, 'relevant') else 1
        for row in frame.itertuples(index=False)
    ])
    if baseline_recalls:
        print('исходный recall@50:', round(float(np.mean(baseline_recalls)), 4))
        print('recall всего пула:', round(float(np.mean(pool_recalls)), 4))
    return {
        'x': np.concatenate(blocks), 'y': np.concatenate(labels),
        'ids': np.concatenate(candidates), 'offsets': np.array(offsets),
        'denominators': denominators, 'names': names,
        'lexical': np.concatenate(lexical_masks), 'hybrid': np.concatenate(hybrid_masks),
    }

посчитаем признаки для обучения и сравнения настроек. в обоих вариантах первые 5000 запросов одинаковые. вторая модель получит все 15000

пока не используем новые 2000 запросов. до них дойдем после выбора настроек


In [8]:
train_candidates = make_features(
    search, rank_queries, 'обучение', item_vectors,
    query_vectors[query_index.get_indexer(rank_queries.search_query)], training=True,
)
validation_candidates = make_features(
    search, validation, 'сравнение', item_vectors,
    query_vectors[query_index.get_indexer(validation.search_query)],
)
print('размер обучающей таблицы:', train_candidates['x'].shape)


обучение:   0%|          | 0/15000 [00:00<?, ?it/s]


обучение:   2%|▏         | 296/15000 [00:20<16:36, 14.76it/s]


обучение:   2%|▏         | 296/15000 [00:30<16:36, 14.76it/s]


обучение:   4%|▍         | 627/15000 [00:40<15:10, 15.78it/s]


обучение:   4%|▍         | 627/15000 [00:50<15:10, 15.78it/s]

...


сравнение:  74%|███████▍  | 1333/1800 [01:36<00:27, 16.84it/s]


сравнение:  94%|█████████▍| 1691/1800 [01:40<00:06, 17.21it/s]


сравнение: 100%|██████████| 1800/1800 [01:46<00:00, 16.91it/s]

исходный recall@50: 0.7974
recall всего пула: 0.96


размер обучающей таблицы: (8040585, 32)


сравним четыре варианта: 5000 и 15000 обучающих запросов, каждый с поиском по смыслу и без него. обычный вариант получит прежние 30 признаков и старый список кандидатов. второй получит расширенный список и все 32 признака

у моделей одинаковая глубина деревьев, скорость обучения и вес правильного ответа. для 5000 запросов проверим 200 и 400 деревьев, а для 15000 еще 600. настройки выберем по recall@50 на тех же 1800 запросах

дополнительно применим обычную модель к расширенному списку. это покажет, помогает ли сам новый список еще до обучения модели на новых признаках


In [9]:
def recall_values(scores, batch, kind):
    values = []
    for i, (start, end) in enumerate(zip(batch['offsets'][:-1], batch['offsets'][1:])):
        positions = start + np.flatnonzero(batch[kind][start:end])
        top = positions[np.argpartition(scores[positions], -50)[-50:]]
        values.append(batch['y'][top].sum() / batch['denominators'][i])
    return np.array(values)


def predict_scores(model, batch, kind, trees):
    width = 30 if kind == 'lexical' else 32
    return model.predict(batch['x'][:, :width], prediction_type='RawFormulaVal', ntree_end=trees)


models = {}
results = []
for count, kind in [(5000, 'lexical'), (15000, 'lexical'), (5000, 'hybrid'), (15000, 'hybrid')]:
    stop = train_candidates['offsets'][count]
    rows = np.flatnonzero(train_candidates[kind][:stop])
    width = 30 if kind == 'lexical' else 32
    trees = 400 if count == 5000 else 600
    print('обучающих запросов:', count, 'поиск:', kind, flush=True)
    model = CatBoostClassifier(
        iterations=trees, depth=6, learning_rate=0.06, l2_leaf_reg=6,
        loss_function='Logloss', class_weights=[1, 30], random_seed=42,
        thread_count=4, verbose=200, allow_writing_files=False,
    )
    model.fit(train_candidates['x'][rows, :width], train_candidates['y'][rows])
    models[(count, kind)] = model
    for limit in range(200, trees + 1, 200):
        scores = predict_scores(model, validation_candidates, kind, limit)
        value = recall_values(scores, validation_candidates, kind).mean()
        results.append({'queries': count, 'search': kind, 'trees': limit, 'recall@50': value})
        print(results[-1], flush=True)
    if kind == 'lexical':
        scores = predict_scores(model, validation_candidates, kind, trees)
        print('та же модель на расширенном списке:', round(recall_values(scores, validation_candidates, 'hybrid').mean(), 4))

comparison = pd.DataFrame(results)
display(comparison)
best = comparison.loc[comparison['recall@50'].idxmax()]
best_count, best_kind, best_trees = int(best.queries), str(best['search']), int(best.trees)
model = models[(best_count, best_kind)]
base_rows = comparison[(comparison.queries == 5000) & (comparison['search'] == 'lexical')]
base_best = base_rows.loc[base_rows['recall@50'].idxmax()]
base_model = models[(5000, 'lexical')]
base_trees = int(base_best.trees)
print('выбрано:', best_count, best_kind, best_trees)
del train_candidates, validation_candidates, rows
_ = gc.collect()

обучающих запросов: 5000 поиск: lexical


0:	learn: 0.5846444	total: 237ms	remaining: 1m 34s


200:	learn: 0.0989366	total: 31.6s	remaining: 31.3s


399:	learn: 0.0904827	total: 1m 1s	remaining: 0us


{'queries': 5000, 'search': 'lexical', 'trees': 200, 'recall@50': np.float64(0.8391666666666666)}


...

{'queries': 15000, 'search': 'hybrid', 'trees': 200, 'recall@50': np.float64(0.8530555555555556)}


{'queries': 15000, 'search': 'hybrid', 'trees': 400, 'recall@50': np.float64(0.8572222222222222)}


{'queries': 15000, 'search': 'hybrid', 'trees': 600, 'recall@50': np.float64(0.8588888888888889)}


,queries,search,trees,recall@50
0,5000,lexical,200,0.839167
1,5000,lexical,400,0.840000
2,15000,lexical,200,0.839259
3,15000,lexical,400,0.844537
4,15000,lexical,600,0.846759
5,5000,hybrid,200,0.851389
6,5000,hybrid,400,0.850833
7,15000,hybrid,200,0.853056
8,15000,hybrid,400,0.857222
9,15000,hybrid,600,0.858889


выбрано: 15000 hybrid 600


настройки выбрали. проверим их на 2000 новых запросов и сравним с моделью на 5000 запросах без поиска по смыслу. у обеих моделей одинаковая история и один корпус объявлений

для каждого запроса сохраним его recall. так можно посчитать не только средний результат, но и сколько запросов стало лучше или хуже. по этой проверке настройки уже не меняем


In [10]:
fresh_candidates = make_features(
    search, fresh, 'новая проверка', item_vectors,
    query_vectors[query_index.get_indexer(fresh.search_query)],
)
base_scores = predict_scores(base_model, fresh_candidates, 'lexical', base_trees)
new_scores = predict_scores(model, fresh_candidates, best_kind, best_trees)
base_values = recall_values(base_scores, fresh_candidates, 'lexical')
new_values = recall_values(new_scores, fresh_candidates, best_kind)
display(pd.DataFrame({
    'вариант': ['5000 без поиска по смыслу', 'выбранный вариант'],
    'recall@50': [base_values.mean(), new_values.mean()],
}))
print('лучше:', int((new_values > base_values).sum()), 'хуже:', int((new_values < base_values).sum()))
del fresh_candidates, base_scores, new_scores
_ = gc.collect()


новая проверка:   0%|          | 0/2000 [00:00<?, ?it/s]


новая проверка:  16%|█▋        | 328/2000 [00:20<01:42, 16.37it/s]


новая проверка:  16%|█▋        | 328/2000 [00:36<01:42, 16.37it/s]


новая проверка:  34%|███▍      | 677/2000 [00:40<01:17, 16.99it/s]


новая проверка:  34%|███▍      | 677/2000 [00:56<01:17, 16.99it/s]

...


новая проверка:  86%|████████▋ | 1728/2000 [01:40<00:15, 17.42it/s]


новая проверка: 100%|██████████| 2000/2000 [01:55<00:00, 17.28it/s]

исходный recall@50: 0.7867
recall всего пула: 0.9525


,вариант,recall@50
0,5000 без поиска по смыслу,0.84322
1,выбранный вариант,0.85172


лучше: 55 хуже: 37


на новых 2000 запросах recall@50 вырос с 0.84322 до 0.85172. у 55 запросов результат стал лучше, у 37 хуже, у остальных не изменился. прирост меньше, чем при выборе настроек, но он сохранился на отдельных данных

еще посмотрим результат на прежних 1200 запросах. они уже использовались в прошлых экспериментах, поэтому это дополнительная проверка, а основной результат выше получен на новых 2000


In [11]:
test_candidates = make_features(
    search, test, 'прежняя проверка', item_vectors,
    query_vectors[query_index.get_indexer(test.search_query)],
)
base_scores = predict_scores(base_model, test_candidates, 'lexical', base_trees)
new_scores = predict_scores(model, test_candidates, best_kind, best_trees)
display(pd.DataFrame({
    'вариант': ['5000 без поиска по смыслу', 'выбранный вариант'],
    'recall@50': [
        recall_values(base_scores, test_candidates, 'lexical').mean(),
        recall_values(new_scores, test_candidates, best_kind).mean(),
    ],
}))
del test_candidates, base_scores, new_scores
_ = gc.collect()


прежняя проверка:   0%|          | 0/1200 [00:00<?, ?it/s]


прежняя проверка:  27%|██▋       | 320/1200 [00:20<00:55, 15.97it/s]


прежняя проверка:  27%|██▋       | 320/1200 [00:38<00:55, 15.97it/s]


прежняя проверка:  55%|█████▍    | 658/1200 [00:40<00:32, 16.51it/s]


прежняя проверка:  55%|█████▍    | 658/1200 [00:58<00:32, 16.51it/s]


прежняя проверка:  83%|████████▎ | 1001/1200 [01:00<00:11, 16.80it/s]


прежняя проверка: 100%|██████████| 1200/1200 [01:11<00:00, 16.68it/s]

исходный recall@50: 0.7981
recall всего пула: 0.9421


,вариант,recall@50
0,5000 без поиска по смыслу,0.842222
1,выбранный вариант,0.853056


для итогового ответа пересоберем поиск по benchmark_items, используя историю из всего train. смысловые векторы этих объявлений уже посчитаны, возьмем нужные строки по item_id

оставим выбранную модель и число деревьев. для каждого запроса сохраним 50 объявлений с самыми большими оценками


In [12]:
benchmark_positions = item_index.get_indexer(benchmark_items.item_id)
assert (benchmark_positions >= 0).all()
benchmark_vectors = np.asarray(item_vectors[benchmark_positions])
del search, all_items
stem.cache_clear()
_ = gc.collect()
final_search = build_search(benchmark_items, interactions)
final_candidates = make_features(
    final_search, queries, 'ответ', benchmark_vectors,
    query_vectors[query_index.get_indexer(queries.search_query)],
)
final_scores = predict_scores(model, final_candidates, best_kind, best_trees)
item_ids = benchmark_items.item_id.to_numpy()
predictions = []
for start, end in zip(final_candidates['offsets'][:-1], final_candidates['offsets'][1:]):
    positions = start + np.flatnonzero(final_candidates[best_kind][start:end])
    top = positions[np.argpartition(final_scores[positions], -50)[-50:]]
    top = top[np.argsort(-final_scores[top])]
    predictions.append(item_ids[final_candidates['ids'][top]].tolist())
answer = pd.DataFrame({
    'query_id': queries.query_id,
    'answer': [' '.join(found) for found in predictions],
})
answer.to_csv('answer.csv', index=False, encoding='utf-8')


слова:   0%|          | 0/189212 [00:00<?, ?it/s]


слова:  37%|███▋      | 70781/189212 [00:20<00:33, 3539.05it/s]


слова:  37%|███▋      | 70781/189212 [00:31<00:33, 3539.05it/s]


слова:  77%|███████▋  | 145952/189212 [00:40<00:11, 3668.15it/s]


слова:  77%|███████▋  | 145952/189212 [00:51<00:11, 3668.15it/s]

...


ответ:   0%|          | 0/2452 [00:00<?, ?it/s]


ответ:  37%|███▋      | 902/2452 [00:20<00:34, 45.07it/s]


ответ:  37%|███▋      | 902/2452 [00:37<00:34, 45.07it/s]


ответ:  75%|███████▌  | 1846/2452 [00:40<00:13, 46.31it/s]


ответ: 100%|██████████| 2452/2452 [00:52<00:00, 46.74it/s]

прочитаем сохраненный answer.csv и проверим, что в нем ровно нужные query_id и каждый встречается один раз. в каждом ответе должно быть не больше 50 разных item_id, и все они должны быть в benchmark_items

еще проверим названия колонок и запись самих идентификаторов. при чтении явно оставим их строками, чтобы не потерять ведущие нули


In [13]:
saved = pd.read_csv('answer.csv', dtype=str, keep_default_na=False, encoding='utf-8')
assert saved.columns.tolist() == ['query_id', 'answer']
assert len(saved) == len(queries)
assert saved.query_id.is_unique
assert saved.query_id.str.len().eq(16).all()
assert set(saved.query_id) == set(queries.query_id)

allowed_ids = set(benchmark_items.item_id)
for value in saved.answer:
    item_ids = value.split(' ')
    assert 1 <= len(item_ids) <= 50
    assert len(item_ids) == len(set(item_ids))
    assert all(re.fullmatch(r'[0-9a-f]{16}', item_id) for item_id in item_ids)
    assert set(item_ids) <= allowed_ids

print('формат проверен:', len(saved), 'строки, по 50 уникальных item_id')
print('файл:', Path('answer.csv').resolve())
print('время выполнения:', round((time.time() - started) / 60, 1), 'мин')
saved.head(3)

формат проверен: 2452 строки, по 50 уникальных item_id
файл: /Applications/programming/vscode_projects/candidate-generation-for-a-service-category/answer.csv
время выполнения: 36.4 мин


,query_id,answer
0,70DfDUpwjxB4lzFd,b92ee8f432cec2d1 d722bcda1a555091 255fbeaf526a...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 422d3ffdd5bbf626 367af128a9ea...
2,LZCZNoVG4AFUkVRJ,dab52187b4500d9b d8fce513e4f000a7 d62074dd39ca...
